In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_2009_2010=pd.read_excel(r"E:\SEM 8\Sales_Intelligence_System\Datasets\Online Retail Dataset\online_retail_II.xlsx", sheet_name="Year 2009-2010")
data_2010_2011=pd.read_excel(r"E:\SEM 8\Sales_Intelligence_System\Datasets\Online Retail Dataset\online_retail_II.xlsx", sheet_name="Year 2010-2011")

In [ ]:
data_2009_2010.head()

In [ ]:
data_2010_2011.head()

In [ ]:
data_2009_2010.shape,data_2010_2011.shape

In [ ]:
merged_data=pd.concat([data_2009_2010,data_2010_2011],ignore_index=True)

In [ ]:
merged_data.head()

In [ ]:
merged_data.shape

In [ ]:
merged_data.describe(include='all')

In [ ]:
merged_data.info()

In [ ]:
merged_data.isnull().sum()

In [ ]:
merged_data.nunique()

In [ ]:
merged_data['Country'].unique()

In [ ]:
merged_data['Country'].value_counts()

In [ ]:
merged_data['Country']=merged_data['Country'].replace({'EIRE':'Ireland','RSA':'South Africa'})

In [ ]:
merged_data['Country'].value_counts()

In [ ]:
merged_data['Description'].unique()

In [ ]:
merged_data['Description'].value_counts()

In [ ]:
merged_data=merged_data.rename(columns={'Description':'Product'})

In [ ]:
merged_data.head()

In [ ]:
merged_data['Invoice'].unique()

In [ ]:
merged_data['Invoice'].value_counts()

In [ ]:
cancelled_data=merged_data[merged_data['Invoice'].astype(str).str.startswith('C')]

In [ ]:
cancelled_data

In [ ]:
merged_data=merged_data[~merged_data['Invoice'].astype(str).str.startswith('C')]

In [ ]:
merged_data.shape,cancelled_data.shape

In [ ]:
merged_data=merged_data.dropna(subset=['Customer ID'])

In [ ]:
merged_data.shape

In [ ]:
merged_data=merged_data[(merged_data['Quantity']>0) & (merged_data['Price']>0)]

In [ ]:
merged_data.shape

In [ ]:
merged_data = merged_data.copy()

In [ ]:
merged_data['Revenue']=merged_data['Quantity']*merged_data['Price']

In [ ]:
merged_data.head()

In [ ]:
merged_data.shape

In [ ]:
merged_data['Revenue'].sum()

In [ ]:
merged_data['Customer ID'].nunique()

In [ ]:
merged_data.groupby('Customer ID')["Invoice"].nunique()

## RFM Analysis

In [ ]:
ref_date=merged_data['InvoiceDate'].max()
ref_date

In [ ]:
rfm=merged_data.groupby('Customer ID').agg({
    'InvoiceDate': lambda x:(ref_date-x.max()).days,
    'Invoice':'nunique',
    'Revenue':'sum'}).reset_index()

In [ ]:
rfm.columns=['CustomerID','Recency','Frequency','Monetary']

In [ ]:
rfm.head()

In [ ]:
rfm.describe()

In [ ]:
rfm[['Recency','Frequency','Monetary']].hist(figsize=(12,4),bins=30)

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

In [ ]:
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)


In [ ]:
rfm['RFM_Score']=( rfm['R_Score']*100 + rfm['F_Score']*10 + rfm['M_Score'])

In [ ]:
rfm.head()

In [ ]:
def segment_customer(row):
    if row['RFM_Score']>=444:
        return 'Champion'
    elif row['F_Score']>=4:
        return 'Loyal Customer'
    elif row['R_Score']<=2 and row['F_Score']>=3:
        return 'At Risk'
    elif row['R_Score']==1:
        return 'Lost Customer'
    else:
        return 'Potential Customer'

In [ ]:
rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['Segment'].value_counts()

In [ ]:
rfm.shape

In [ ]:
pd.options.display.float_format = '{:,.0f}'.format

In [ ]:
rfm.groupby('Segment').agg({
    'CustomerID':'count',
    'Monetary':'sum',
    'Frequency':'mean'}).sort_values(by='Monetary',ascending=False)

In [ ]:
rfm['Segment'].value_counts().plot(kind='bar',figsize=(8,4),title='Customer Segmentation')
plt.xticks(rotation=10, ha='right')
plt.ylabel("Number of Customers")
plt.savefig(r'E:\SEM 8\Sales_Intelligence_System\Code_Implementation\Online_retail\Customer_segmentation.png')

In [ ]:
rfm.to_csv(r"E:\SEM 8\Sales_Intelligence_System\Code_Implementation\Online_retail\rfm_analysis.csv",index=False)

In [ ]:
segment_revenue = (
    rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False))
plt.figure()
segment_revenue.plot(kind='bar',figsize=(8,4),title='Revenue Contribution by Customer Segmentation')
plt.ylabel("Total Revenue")
plt.xlabel("Customer Segment")
plt.ticklabel_format(style='plain', axis='y')
plt.xticks(rotation=10, ha='right')
plt.savefig(r'E:\SEM 8\Sales_Intelligence_System\Code_Implementation\Online_retail\Revenue_Contribution.png')

In [ ]:
rfm.groupby('Segment').agg({
    'CustomerID': 'count',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(2).sort_values(by='Monetary', ascending=False)


In [ ]:
rfm['RFM_Score'].value_counts().head(10).plot(kind='bar',figsize=(8,4),title="Top RFM Score Combinations")
plt.savefig(r'E:\SEM 8\Sales_Intelligence_System\Code_Implementation\Online_retail\top_rfm_combination.png')